In [1]:
import torch
import pandas as pd
import numpy as np
import ta
import yfinance as yf

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
print(f"GPU      : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Pandas   : {pd.__version__}")
print(f"yfinance : {yf.__version__}")
print()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device utilisé : {device}")

torch.manual_seed(42)
np.random.seed(42)
print("\nPret pour le module 4 LSTM !")

PyTorch  : 2.5.1+cu121
CUDA     : True
GPU      : NVIDIA GeForce RTX 4060 Laptop GPU
Pandas   : 2.3.3
yfinance : 1.3.0

Device utilisé : cuda

Pret pour le module 4 LSTM !


In [2]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, classification_report
import yfinance as yf
import ta
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
np.random.seed(42)

# Télécharger les données
print("Téléchargement des données SPY...")
df = yf.download("SPY", start="2018-01-01", progress=False)
df.columns = [col[0] for col in df.columns]

# Calcul des features avec la librairie ta
df['Return']         = df['Close'].pct_change() * 100
df['RSI']            = ta.momentum.RSIIndicator(df['Close'], window=14).rsi()
df['MACD']           = ta.trend.MACD(df['Close']).macd()
df['MACD_signal']    = ta.trend.MACD(df['Close']).macd_signal()
df['BB_high']        = ta.volatility.BollingerBands(df['Close']).bollinger_hband()
df['BB_low']         = ta.volatility.BollingerBands(df['Close']).bollinger_lband()
df['Volatility_20d'] = df['Return'].rolling(20).std()
df['Return_5d']      = df['Close'].pct_change(5) * 100
df['Return_20d']     = df['Close'].pct_change(20) * 100
df['Volume_ratio']   = df['Volume'] / df['Volume'].rolling(20).mean()
df['MM50']           = df['Close'].rolling(50).mean()
df['MM200']          = df['Close'].rolling(200).mean()
df['MM_signal']      = (df['MM50'] > df['MM200']).astype(int)

# Label : 1 si hausse demain, 0 sinon
df['Label'] = (df['Close'].shift(-1) > df['Close']).astype(int)

# Nettoyer
df = df.dropna()

print(f"Données chargées    : {len(df)} jours")
print(f"Période             : {df.index[0].date()} → {df.index[-1].date()}")
print(f"Hausse / Baisse     : {df['Label'].sum()} / {(df['Label']==0).sum()}")
print(f"Features disponibles: {len(df.columns)-1}")

Téléchargement des données SPY...
Données chargées    : 1890 jours
Période             : 2018-10-16 → 2026-04-24
Hausse / Baisse     : 1045 / 845
Features disponibles: 18


In [7]:
from sklearn.preprocessing import RobustScaler
from torch.utils.data import DataLoader, TensorDataset

SEQ_LEN  = 60
features = ['RSI', 'MACD', 'MACD_signal', 'BB_high', 'BB_low',
            'Volatility_20d', 'Return_5d', 'Return_20d',
            'Volume_ratio', 'MM_signal', 'Return']

# RobustScaler — meilleur que MinMaxScaler pour données financières
# car il est insensible aux valeurs extrêmes (crashs, pics)
scaler   = RobustScaler()

# IMPORTANT — on scale séparément train et test pour éviter le data leakage
split_idx = int(len(df) * 0.8)
df_train  = df.iloc[:split_idx]
df_test   = df.iloc[split_idx:]

X_train_raw = scaler.fit_transform(df_train[features])
X_test_raw  = scaler.transform(df_test[features])
y_train_raw = df_train['Label'].values
y_test_raw  = df_test['Label'].values

# Créer les séquences
def create_sequences(X, y, seq_len):
    Xs, ys = [], []
    for i in range(seq_len, len(X)):
        Xs.append(X[i-seq_len:i])
        ys.append(y[i])
    return np.array(Xs), np.array(ys)

X_train_seq, y_train_seq = create_sequences(X_train_raw, y_train_raw, SEQ_LEN)
X_test_seq,  y_test_seq  = create_sequences(X_test_raw,  y_test_raw,  SEQ_LEN)

# Convertir en tenseurs
X_train = torch.FloatTensor(X_train_seq).to(device)
X_test  = torch.FloatTensor(X_test_seq).to(device)
y_train = torch.FloatTensor(y_train_seq).to(device)
y_test  = torch.FloatTensor(y_test_seq).to(device)

# Dates pour les graphiques
dates_test = df_test.index[SEQ_LEN:]

print(f"Normalisation       : RobustScaler")
print(f"X_train shape       : {X_train.shape}")
print(f"X_test  shape       : {X_test.shape}")
print(f"Période test        : {dates_test[0].date()} → {dates_test[-1].date()}")
print(f"Hausse train        : {y_train_seq.mean()*100:.1f}%")
print(f"Hausse test         : {y_test_seq.mean()*100:.1f}%")

Normalisation       : RobustScaler
X_train shape       : torch.Size([1452, 60, 11])
X_test  shape       : torch.Size([318, 60, 11])
Période test        : 2025-01-17 → 2026-04-24
Hausse train        : 55.4%
Hausse test         : 55.7%


In [4]:
class LSTMTrader(nn.Module):
    def __init__(self, input_size, hidden_size1, hidden_size2, dropout):
        super(LSTMTrader, self).__init__()

        # Couche 1 — capture patterns courts (3-10 jours)
        self.lstm1 = nn.LSTM(
            input_size  = input_size,
            hidden_size = hidden_size1,
            num_layers  = 1,
            batch_first = True
        )

        # Couche 2 — capture patterns longs (10-60 jours)
        self.lstm2 = nn.LSTM(
            input_size  = hidden_size1,
            hidden_size = hidden_size2,
            num_layers  = 1,
            batch_first = True
        )

        self.dropout   = nn.Dropout(dropout)
        self.norm1     = nn.LayerNorm(hidden_size1)
        self.norm2     = nn.LayerNorm(hidden_size2)

        # Couche de décision finale
        self.fc1       = nn.Linear(hidden_size2, 32)
        self.fc2       = nn.Linear(32, 1)
        self.sigmoid   = nn.Sigmoid()
        self.relu      = nn.ReLU()

    def forward(self, x):
        # Passage couche 1
        out1, _  = self.lstm1(x)
        out1     = self.norm1(out1)
        out1     = self.dropout(out1)

        # Passage couche 2
        out2, _  = self.lstm2(out1)
        out2     = self.norm2(out2)
        out2     = self.dropout(out2)

        # On prend uniquement le dernier timestep
        out      = out2[:, -1, :]

        # Couches de décision
        out      = self.relu(self.fc1(out))
        out      = self.dropout(out)
        out      = self.sigmoid(self.fc2(out))

        return out.squeeze()

# Instancier le modèle
model = LSTMTrader(
    input_size  = len(features),  # 11 features
    hidden_size1 = 128,
    hidden_size2 = 64,
    dropout      = 0.3
).to(device)

# Résumé du modèle
total_params = sum(p.numel() for p in model.parameters())
print(f"Architecture LSTM :")
print(f"  Entrée        : {len(features)} features × 60 jours")
print(f"  LSTM 1        : 128 unités cachées")
print(f"  LSTM 2        : 64 unités cachées")
print(f"  Dropout       : 30%")
print(f"  Dense         : 64 → 32 → 1")
print(f"  Sortie        : Sigmoid (0 à 1 = probabilité hausse)")
print(f"\nParamètres totaux : {total_params:,}")
print(f"Device            : {next(model.parameters()).device}")

Architecture LSTM :
  Entrée        : 11 features × 60 jours
  LSTM 1        : 128 unités cachées
  LSTM 2        : 64 unités cachées
  Dropout       : 30%
  Dense         : 64 → 32 → 1
  Sortie        : Sigmoid (0 à 1 = probabilité hausse)

Paramètres totaux : 124,353
Device            : cuda:0


In [5]:
from torch.utils.data import DataLoader, TensorDataset

# Hyperparamètres
EPOCHS      = 100
BATCH_SIZE  = 64
LR          = 0.001

# Dataset et DataLoader
train_dataset = TensorDataset(X_train, y_train)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Poids pour compenser le déséquilibre hausse/baisse
n_baisse  = (y_seq[:split] == 0).sum()
n_hausse  = (y_seq[:split] == 1).sum()
pos_weight = torch.tensor([n_baisse / n_hausse]).to(device)

# Loss et optimiseur
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=10, factor=0.5, verbose=False
)

# Historique
history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
best_acc    = 0
best_model  = None
patience    = 20
no_improve  = 0

print(f"Entraînement sur {EPOCHS} epochs — RTX 4060 CUDA")
print(f"Batch size : {BATCH_SIZE} | LR : {LR}")
print("-" * 55)

for epoch in range(EPOCHS):
    # MODE ENTRAÎNEMENT
    model.train()
    train_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        output = model(X_batch)
        loss   = criterion(output, y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)

    # MODE ÉVALUATION
    model.eval()
    with torch.no_grad():
        val_output = model(X_test)
        val_loss   = criterion(val_output, y_test).item()
        val_pred   = (val_output > 0.5).float()
        val_acc    = (val_pred == y_test).float().mean().item()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    scheduler.step(val_loss)

    # Sauvegarder le meilleur modèle
    if val_acc > best_acc:
        best_acc   = val_acc
        best_model = {k: v.clone() for k, v in model.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1

    # Afficher progression
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | "
              f"Loss train: {train_loss:.4f} | "
              f"Loss val: {val_loss:.4f} | "
              f"Acc val: {val_acc*100:.1f}% | "
              f"Meilleur: {best_acc*100:.1f}%")

    # Early stopping
    if no_improve >= patience:
        print(f"\nEarly stopping à l'epoch {epoch+1}")
        break

# Charger le meilleur modèle
model.load_state_dict(best_model)
print(f"\nEntraînement terminé !")
print(f"Meilleure précision : {best_acc*100:.1f}%")

Entraînement sur 100 epochs — RTX 4060 CUDA
Batch size : 64 | LR : 0.001
-------------------------------------------------------
Epoch  10/100 | Loss train: 0.6197 | Loss val: 0.6179 | Acc val: 43.7% | Meilleur: 43.7%
Epoch  20/100 | Loss train: 0.6192 | Loss val: 0.6179 | Acc val: 43.7% | Meilleur: 43.7%

Early stopping à l'epoch 21

Entraînement terminé !
Meilleure précision : 43.7%


In [9]:
from torch.utils.data import DataLoader, TensorDataset

# Hyperparamètres optimisés
EPOCHS     = 200
BATCH_SIZE = 32
LR         = 0.0003
patience   = 40

# Modèle réinitialisé avec architecture plus profonde
model = LSTMTrader(
    input_size   = len(features),
    hidden_size1 = 128,
    hidden_size2 = 64,
    dropout      = 0.25
).to(device)

criterion = nn.BCELoss()
optimizer = torch.optim.AdamW(model.parameters(),
                              lr=LR,
                              weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr     = LR,
    epochs     = EPOCHS,
    steps_per_epoch = len(DataLoader(
        TensorDataset(X_train, y_train),
        batch_size=BATCH_SIZE))
)

history    = {'train_loss': [], 'val_loss': [], 'val_acc': []}
best_acc   = 0
best_model = None
patience   = 30
no_improve = 0

print(f"Entraînement corrigé — {EPOCHS} epochs")
print(f"Batch: {BATCH_SIZE} | LR: {LR} | Dropout: 0.2")
print("-" * 60)

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        output     = model(X_batch)
        loss       = criterion(output, y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)
    scheduler.step()

    model.eval()
    with torch.no_grad():
        val_output = model(X_test)
        val_loss   = criterion(val_output, y_test).item()
        val_pred   = (val_output > 0.5).float()
        val_acc    = (val_pred == y_test).float().mean().item()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    if val_acc > best_acc:
        best_acc   = val_acc
        best_model = {k: v.clone() 
                      for k, v in model.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1

    if (epoch + 1) % 15 == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | "
              f"Train: {train_loss:.4f} | "
              f"Val: {val_loss:.4f} | "
              f"Acc: {val_acc*100:.1f}% | "
              f"Best: {best_acc*100:.1f}%")

    if no_improve >= patience:
        print(f"\nEarly stopping epoch {epoch+1}")
        break

model.load_state_dict(best_model)
print(f"\nMeilleure précision : {best_acc*100:.1f}%")

Entraînement corrigé — 200 epochs
Batch: 32 | LR: 0.0003 | Dropout: 0.2
------------------------------------------------------------
Epoch  15/200 | Train: 0.6872 | Val: 0.6915 | Acc: 51.3% | Best: 52.5%
Epoch  30/200 | Train: 0.6873 | Val: 0.6919 | Acc: 51.6% | Best: 52.5%

Early stopping epoch 32

Meilleure précision : 52.5%
